# Telco Customer Churn: EDA & Baseline Model

**How to run:** put `WA_Fn-UseC_-Telco-Customer-Churn.csv` next to this notebook (or edit `DATA_PATH`), then *Run all*.
Requires: `pandas numpy matplotlib seaborn scikit-learn`.

Sections: 0 Setup, 1 Load & inspect, 2 Clean, 3 Univariate, 4 Bivariate, 5 Feature engineering, 6 Baseline model, 7 Conclusions.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"   # Colab: "/content/WA_Fn-UseC_-Telco-Customer-Churn.csv"
RANDOM_STATE = 42

## 1. Load and inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
display(df.describe())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())
print("\nMissing values per column (NaN):")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Hidden missing values: TotalCharges is 'object' because of blank strings
blank_mask = df["TotalCharges"].str.strip() == ""
print("Blank TotalCharges:", blank_mask.sum())
df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

## 2. Clean and preprocess

In [ ]:
df_clean = df.copy()

# 1) Fix TotalCharges type (blanks -> NaN). All blanks have tenure == 0.
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
print("NaNs in TotalCharges:", df_clean["TotalCharges"].isna().sum())
print("Tenure of those rows:", df_clean.loc[df_clean["TotalCharges"].isna(), "tenure"].unique())

# 2) Imputation is deferred to the model pipeline (section 6) to avoid leakage.
#    For EDA only, fill with 0 (nothing billed yet).
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

# 3) Target to 0/1
df_clean["Churn"] = df_clean["Churn"].map({"No": 0, "Yes": 1})

# 4) SeniorCitizen as readable label (for plots)
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 5) Merge 'No internet service' / 'No phone service' into 'No'
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
for c in addon_cols + ["MultipleLines"]:
    df_clean[c] = df_clean[c].replace({"No internet service": "No", "No phone service": "No"})

# 6) Drop identifier
df_clean = df_clean.drop(columns="customerID")

df_clean.dtypes.value_counts()

## 3. Univariate analysis

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in df_clean.columns if c not in num_cols + ["Churn"]]

# Target balance
ax = sns.countplot(data=df_clean, x="Churn")
ax.set_xticklabels(["No (0)", "Yes (1)"])
ax.set_title(f"Churn distribution (churn rate = {df_clean['Churn'].mean():.1%})")
plt.show()

In [ ]:
# Numeric: histograms + KDE
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df_clean[col], kde=True, bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical: count plots in a grid
n = len(cat_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.6 * nrows))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(data=df_clean, x=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Bivariate analysis: churn vs key features

In [ ]:
# Numeric vs churn: boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df_clean, x="Churn", y=col, ax=ax)
    ax.set_xticklabels(["No", "Yes"])
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

df_clean.groupby("Churn")[num_cols].agg(["mean", "median"]).round(1)

In [ ]:
# Churn rate by categorical feature (the most useful plot in this project)
def churn_rate_plot(col, ax=None, order=None):
    rate = df_clean.groupby(col)["Churn"].mean().mul(100)
    if order: rate = rate.reindex(order)
    ax = ax or plt.gca()
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.axhline(df_clean["Churn"].mean() * 100, color="red", ls="--", label="overall")
    ax.set_ylabel("Churn rate (%)"); ax.set_xlabel(""); ax.set_title(f"Churn rate by {col}")
    ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
churn_rate_plot("Contract", axes[0], ["Month-to-month", "One year", "Two year"])
churn_rate_plot("InternetService", axes[1])
churn_rate_plot("PaymentMethod", axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by other features (loop)
other = ["OnlineSecurity", "TechSupport", "PaperlessBilling", "Partner", "Dependents", "SeniorCitizen"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), other):
    churn_rate_plot(col, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Tenure: churn rate by month (shows early-life risk)
tenure_rate = df_clean.groupby("tenure")["Churn"].mean()
plt.figure(figsize=(10, 4))
sns.lineplot(x=tenure_rate.index, y=tenure_rate.values)
plt.title("Churn rate by tenure (months)"); plt.ylabel("Churn rate"); plt.show()

In [ ]:
# Correlation heatmap (numeric + 0/1 encoded for correlation only)
corr_df = df_clean[num_cols + ["Churn"]].copy()
for c in ["Partner", "Dependents", "PaperlessBilling", "PhoneService"]:
    corr_df[c] = df_clean[c].map({"No": 0, "Yes": 1})
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap"); plt.show()

## 5. Feature engineering

In [ ]:
fe = df_clean.copy()

# Tenure bins
fe["tenure_bin"] = pd.cut(fe["tenure"], bins=[-1, 12, 24, 48, 72],
                          labels=["0-12", "13-24", "25-48", "49-72"])

# Family vs single
fe["has_family"] = ((fe["Partner"] == "Yes") | (fe["Dependents"] == "Yes")).astype(int)

# Number of add-on services
fe["n_addons"] = (fe[addon_cols] == "Yes").sum(axis=1)

# Average monthly spend (guard tenure == 0)
fe["avg_monthly"] = np.where(fe["tenure"] > 0, fe["TotalCharges"] / fe["tenure"], fe["MonthlyCharges"])

# Auto-pay flag
fe["autopay"] = fe["PaymentMethod"].str.contains("automatic").astype(int)

# Interaction: month-to-month x fiber
fe["m2m_fiber"] = ((fe["Contract"] == "Month-to-month") & (fe["InternetService"] == "Fiber optic")).astype(int)

# Churn rate by new segments
for col in ["tenure_bin", "has_family", "n_addons", "autopay", "m2m_fiber"]:
    print(fe.groupby(col, observed=True)["Churn"].agg(["mean", "count"]).round(3), "\n")

In [ ]:
# Segment heatmap: Contract x InternetService
pivot = fe.pivot_table(index="Contract", columns="InternetService", values="Churn", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Reds")
plt.title("Churn rate: Contract x InternetService"); plt.show()

## 6. Baseline model

Rules: split first, stratify, put imputation, scaling and encoding **inside a Pipeline**, and compare to a dummy baseline.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)

# Use the raw-ish frame: TotalCharges NaN restored so the pipeline imputes it from train data only
data = df_clean.drop(columns=[]).copy()
data["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = data.drop(columns="Churn")
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in X.columns if c not in num_features]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features),
])

# Dummy baseline
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Majority-class baseline accuracy: {accuracy_score(y_test, dummy.predict(X_test)):.3f}")

# Logistic regression
clf = Pipeline([("prep", preprocess),
                ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]
print(f"Logistic regression accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}\n")
print(classification_report(y_test, pred, target_names=["No churn", "Churn"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["No", "Yes"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

# 5-fold cross-validation (more reliable than one split)
cv = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")
print(f"CV ROC-AUC: {cv.mean():.3f} +/- {cv.std():.3f}")

In [ ]:
# Interpret: largest coefficients
names = clf.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(clf.named_steps["model"].coef_[0], index=names).sort_values()
top = pd.concat([coefs.head(10), coefs.tail(10)])
top.plot(kind="barh", figsize=(8, 7), title="Top coefficients (negative = lowers churn risk)")
plt.tight_layout(); plt.show()

In [ ]:
# Exercise: swap in other models and compare
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

for name, m in {"KNN (k=5)": KNeighborsClassifier(5),
                "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced")}.items():
    p = Pipeline([("prep", preprocess), ("model", m)])
    s = cross_val_score(p, X, y, cv=5, scoring="roc_auc")
    print(f"{name}: CV ROC-AUC = {s.mean():.3f}")

## 7. Conclusions (fill in)

1. Key churn drivers observed: ...
2. Baseline vs dummy: ...
3. Business recommendations: ...
4. Limitations and next steps (tuning, SHAP, threshold selection, cost-sensitive evaluation): ...